# LAB 08 — Pretrained Transformer (BERT) + Fine-tuning trên IMDb
Môn NLP & ứng dụng, HK I 2026 · **From Training a Transformer to Using a Pretrained Language Model**

Các run train nặng (BERT fine-tuning, frozen, from-scratch, data-efficiency, sequence length) được chạy bởi
`pretrained_classification.py` và lưu vào `outputs/`. Notebook này: (1) giải thích pipeline + classification head,
(2) đọc kết quả và trả lời Exp 1–5, (3) Application: metrics + confusion matrix, (4) confidence analysis, (5) dữ liệu error analysis.
Các ô ✍️ là phần nhận xét / diễn giải: **observation → evidence → interpretation**.

In [1]:
import json, sys, warnings, logging
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
warnings.filterwarnings("ignore"); logging.getLogger("transformers").setLevel(logging.ERROR)
sys.path.insert(0, ".")
import pretrained_classification as pc

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
BLUE, ORANGE, GREEN, RED, GREY = "#4C78A8", "#F58518", "#54A24B", "#E45756", "#8C8C8C"
OUT = pc.OUT
res = {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in OUT.glob("*.json") if "metrics" in json.loads(p.read_text(encoding="utf-8"))}
probs = lambda name: np.load(OUT / f"{name}_probs.npz")
print(len(res), "runs:", sorted(res))
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

16 runs: ['bert_random_len128_100', 'frozen_len128_10', 'frozen_len128_100', 'frozen_len128_25', 'frozen_len128_50', 'ft_len128_10', 'ft_len128_100', 'ft_len128_10_ep10', 'ft_len128_25', 'ft_len128_50', 'ft_len256_50', 'ft_len64_50', 'scratch_small_len128_10', 'scratch_small_len128_100', 'scratch_small_len128_25', 'scratch_small_len128_50']
torch 2.14.1+cu126 | cuda: True


## 0. Reproducibility (đề mục 23)

In [2]:
r = res["ft_len128_100"]
repro = {
 "Model": f"{pc.MODEL_NAME} ({r['n_params']:,} params, 6 layer, d=768)", "Tokenizer": f"{pc.MODEL_NAME} WordPiece (fast), vocab 30,522 (trùng vocab bert-base-uncased)",
 "Dataset": "IMDb (stanfordnlp/imdb); domain shift: amazon_polarity (test, 2,000 mẫu), rotten_tomatoes (test, 1,066)",
 "Train split": f"{r['n_train']:,} (subset stratified, seed 42, của 20,000 train trong split LAB 05/06; Exp 3 lấy 10/25/50/100% của pool này)",
 "Validation split": f"{r['n_val']:,} (subset stratified của 5,000 val LAB 05/06)", "Test split": f"{r['n_test']:,} (subset stratified, seed 42, của IMDb test chính thức 25,000)",
 "Max sequence length": "128 (mặc định); Exp 4: 64 / 128 / 256 (truncation giữ phần đầu review)",
 "Batch size": "32 (DistilBERT: micro-batch 4 x 4 accumulation; len 256: 4 x 8; Transformer nhỏ: 64)",
 "Learning rate": "BERT fine-tune 2e-5 (warmup 10% + linear decay); frozen head 1e-3; BERT random-init 1e-4; Transformer nhỏ 5e-4",
 "Epochs": "2 (DistilBERT: fine-tune / frozen / random-init); Transformer nhỏ 6; chọn epoch có val loss nhỏ nhất", "Optimizer": "AdamW (weight decay 0.01), grad clip 1.0, fp16 autocast",
 "Random seed": "42 (1 seed duy nhất cho mọi run)", "Hardware": r["hardware"], "Training time (DistilBERT fine-tune, T=128; GPU dùng chung nên chỉ mang tính tương đối)": f"{r['train_time_s']:.0f} s",
}
display(pd.Series(repro, name="value").to_frame())

,value
Model,"distilbert-base-uncased (66,364,418 params, 6 ..."
Tokenizer,"distilbert-base-uncased WordPiece (fast), voca..."
Dataset,IMDb (stanfordnlp/imdb); domain shift: amazon_...
Train split,"10,000 (subset stratified, seed 42, của 20,000..."
Validation split,"2,000 (subset stratified của 5,000 val LAB 05/06)"
Test split,"5,000 (subset stratified, seed 42, của IMDb te..."
Max sequence length,128 (mặc định); Exp 4: 64 / 128 / 256 (truncat...
Batch size,32 (DistilBERT: micro-batch 4 x 4 accumulation...
Learning rate,BERT fine-tune 2e-5 (warmup 10% + linear decay...
Epochs,2 (DistilBERT: fine-tune / frozen / random-ini...


## 1. Pipeline: Tokenizer → Pretrained Transformer → [CLS] → Linear → Softmax

In [3]:
tok = pc.AutoTokenizer.from_pretrained(pc.MODEL_NAME)
tokens = tok(["This movie is great", "Not good at all, I wanted my money back"], padding=True, truncation=True, max_length=16, return_tensors="pt")
print("tokens.keys():", list(tokens.keys()))
print("input_ids     :\n", tokens["input_ids"])
print("attention_mask:\n", tokens["attention_mask"])
model = pc.CLSClassifier()       # trên CPU, chỉ để minh họa kiến trúc / số tham số
model.eval()
with torch.no_grad():
    logits = model(tokens["input_ids"], tokens["attention_mask"])
print("logits (head CHƯA train, ngẫu nhiên):", logits.numpy().round(3).tolist(), "-> softmax:", torch.softmax(logits, -1).numpy().round(3).tolist())
print("\nClassification head:", model.head, "| dropout:", model.dropout)
tot, trn = pc.count_params(model)
print(f"Fine-tuning : tổng {tot:,} | trainable {trn:,}")
frozen = pc.CLSClassifier(freeze=True)
tot, trn = pc.count_params(frozen)
print(f"Frozen      : tổng {tot:,} | trainable {trn:,}  ({100*trn/tot:.4f}% — chỉ head Linear 768x2+2)")
small = pc.SmallTransformer(tok.vocab_size)
print(f"Small scratch: tổng {pc.count_params(small)[0]:,} (4 layer, d=256, 4 head)")
del model, frozen, small

tokens.keys(): ['input_ids', 'token_type_ids', 'attention_mask']
input_ids     :
 tensor([[ 101, 2023, 3185, 2003, 2307,  102,    0,    0,    0,    0,    0,    0],
        [ 101, 2025, 2204, 2012, 2035, 1010, 1045, 2359, 2026, 2769, 2067,  102]])
attention_mask:
 tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


logits (head CHƯA train, ngẫu nhiên): [[0.04600000008940697, -0.007000000216066837], [0.01600000075995922, -0.1340000033378601]] -> softmax: [[0.5130000114440918, 0.4869999885559082], [0.5370000004768372, 0.46299999952316284]]

Classification head: Linear(in_features=768, out_features=2, bias=True) | dropout: Dropout(p=0.1, inplace=False)
Fine-tuning : tổng 66,364,418 | trainable 66,364,418


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Frozen      : tổng 66,364,418 | trainable 1,538  (0.0023% — chỉ head Linear 768x2+2)
Small scratch: tổng 11,104,770 (4 layer, d=256, 4 head)


✍️ **Nhận xét.** Pipeline đúng như đề: văn bản → tokenizer (`input_ids`, `attention_mask`) → DistilBERT (BERT-style, 6 layer) → vector `[CLS]` (vị trí 0 của `last_hidden_state`) → Dropout → Linear(768→2) → softmax.
`input_ids` là ID token đưa vào embedding; `attention_mask` = 1 cho token thật, 0 cho `[PAD]` (câu thứ nhất bị pad 6 vị trí). Tokenizer trả thêm `token_type_ids` (segment A/B, toàn 0 vì chỉ có 1 câu) nhưng DistilBERT không dùng nó nên code chỉ truyền `input_ids` và `attention_mask` vào model.
Head chưa train cho logits ≈ nhau (softmax ≈ 0.5) → xác nhận head là phần **task-specific, khởi tạo ngẫu nhiên**, còn encoder đã mang kiến thức pretrained.
Ở chế độ **frozen** chỉ có 1,538 tham số (768×2+2) trainable (0.0023%) so với 66.4M của fine-tuning.

## 2. Experiment 1 — Pretrained vs From Scratch

In [4]:
def row(name, label):
    r = res[name]; m = r["metrics"]["imdb_test"]
    return dict(Model=label, Accuracy=m["accuracy"], F1=m["f1"], Precision=m["precision"], Recall=m["recall"], **{"Train time (s)": r["train_time_s"], "Epochs": r["epochs"], "# Params": r["n_params"], "Peak GPU mem (MB)": r["peak_gpu_mem_mb"], "Val acc": r["metrics"]["imdb_val"]["accuracy"], "Best epoch": r["best_epoch"]})
exp1 = pd.DataFrame([
    row("scratch_small_len128_100", "Transformer from scratch (4L, d=256, ~11M)"),
    row("bert_random_len128_100", "DistilBERT kiến trúc, random init (control)"),
    row("ft_len128_100", "Pretrained DistilBERT + fine-tuning"),
])
display(exp1.round(4))

,Model,Accuracy,F1,Precision,Recall,Train time (s),Epochs,# Params,Peak GPU mem (MB),Val acc,Best epoch
0,"Transformer from scratch (4L, d=256, ~11M)",0.7484,0.7328,0.7812,0.6900,386.6025,6,11104770,546.7998,0.7665,3
1,"DistilBERT kiến trúc, random init (control)",0.8000,0.7976,0.8074,0.7880,338.2086,2,66364418,1579.3779,0.8135,1
2,Pretrained DistilBERT + fine-tuning,0.8704,0.8701,0.8719,0.8684,1117.5615,2,66364418,1354.3623,0.8645,1


✍️ **Observation → Evidence → Interpretation (Exp 1).**
- *Observation:* trên cùng split, cùng test subset 5,000 review và cùng `max_len=128`: Transformer nhỏ from scratch đạt acc **0.748** / F1 0.733; DistilBERT cùng kiến trúc nhưng random init đạt **0.800** / 0.798; DistilBERT pretrained + fine-tuning đạt **0.870** / 0.870.
- *Evidence:* khoảng cách pretrained − random-init (cùng kiến trúc, cùng số tham số 66.4M, cùng 2 epoch) là **+7.0 điểm**; pretrained − small-scratch là **+12.2 điểm**. Với n=5,000, sai số chuẩn của accuracy ≈ 0.5 điểm (95% CI ≈ ±1 điểm) nên các khoảng cách này lớn hơn nhiều nhiễu lấy mẫu của test set; nhưng mỗi cấu hình chỉ chạy **1 seed** nên nhiễu do khởi tạo/thứ tự dữ liệu chưa được ước lượng.
- *Interpretation:* vì cặp (random-init, pretrained) giống hệt về kiến trúc và hyper-parameter chính, phần chênh lệch +7 điểm gán được cho **trọng số pretrained** (không phải do kiến trúc hay kích thước). Chênh lệch giữa small-scratch và random-init (+5.2) cho thấy dung lượng model/số epoch cũng có ảnh hưởng. Chưa thể nói cả hai model from-scratch đã được tune tối ưu (lr chỉ thử 1 giá trị).
- **1. Representation khác nhau thế nào?** From scratch: embedding và các lớp attention bắt đầu ngẫu nhiên, chỉ học từ 10,000 review → biểu diễn nghèo (nhiều từ hiếm chỉ xuất hiện vài lần). Pretrained: embedding và các lớp đã mã hóa thông tin từ vựng/cú pháp/ngữ cảnh từ pretraining nên fine-tune chỉ cần chỉnh nhẹ. Bằng chứng gián tiếp: ở epoch 1, DistilBERT pretrained đạt val acc 0.8645 so với 0.8135 của random-init cùng kiến trúc, và frozen encoder (không học gì ở encoder) vẫn đạt 0.79 > 0.75 của small-scratch được train toàn bộ.
- **2. Training data cần khác thế nào?** Xem Exp 3: pretrained giữ 0.797 chỉ với 1,000 review, trong khi small-scratch rơi xuống 0.595 (random-init cùng kiến trúc chỉ được chạy ở 100% dữ liệu nên chưa có đường cong riêng).
- **3. Chi phí tính toán khác thế nào?** Số tham số: 66.4M (DistilBERT) vs 11.1M (small) — gấp ~6 lần; bộ nhớ đỉnh khi train 1.35–1.58 GB vs 0.55 GB. **Cột Train time không so sánh được**: GPU bị dùng chung với các tiến trình khác trong lúc chạy (throughput dao động từ 3 đến ~60 mẫu/giây), nên 1,118 s của run pretrained và 338 s của run random-init (cùng kiến trúc, cùng số bước!) chủ yếu phản ánh mức tranh chấp GPU chứ không phải chi phí thật. Chi phí thật mỗi bước của pretrained và random-init là như nhau (cùng kiến trúc); phần *chi phí ẩn* của pretrained là lượng tính toán pretraining (đã được trả bởi người khác) — thứ không xuất hiện trong bảng.
- **4. Có phù hợp với prediction không?** Chiều thì đúng (pretrained > from scratch, Prediction 1). Độ lớn thì sai: mình đã dự đoán BERT-base ≈ 90–93% và scratch ≈ 82–87%; thực tế 87.0% và 74.8% — cả hai thấp hơn dự đoán vì dùng DistilBERT, chỉ 10,000 review huấn luyện và cắt 128 token (88% review bị cắt, xem `tokenization.ipynb` mục 5), và khoảng cách thực tế (12 điểm so với small scratch) ở cuối khoảng dự đoán.

## 3. Experiment 2 — Feature extraction (frozen) vs Fine-tuning

In [5]:
exp2 = pd.DataFrame([row("frozen_len128_100", "Frozen"), row("ft_len128_100", "Fine-tuning")])
exp2.insert(1, "Encoder trainable?", ["No", "Yes"]); exp2.insert(2, "Trainable params", [res["frozen_len128_100"]["trainable_params"], res["ft_len128_100"]["trainable_params"]])
display(exp2[["Model", "Encoder trainable?", "Trainable params", "Accuracy", "F1", "Train time (s)", "Peak GPU mem (MB)", "Val acc"]].round(4))

,Model,Encoder trainable?,Trainable params,Accuracy,F1,Train time (s),Peak GPU mem (MB),Val acc
0,Frozen,No,1538,0.7918,0.7889,378.2882,439.6396,0.8025
1,Fine-tuning,Yes,66364418,0.8704,0.8701,1117.5615,1354.3623,0.8645


✍️ **Observation → Evidence → Interpretation (Exp 2).**
- *Observation:* frozen đạt acc **0.792** / F1 0.789; fine-tuning đạt **0.870** / 0.870 — frozen kém hơn **7.9 điểm** (chỉ train 1,538 tham số thay vì 66.4M). Val acc cùng chiều (0.8025 vs 0.8645).
- **Vì sao frozen encoder vẫn phân loại được?** Pretraining (MLM + distillation) đã buộc encoder mã hóa nghĩa của từ và ngữ cảnh, trong đó có thông tin liên quan sentiment (các từ như *great/terrible* và ngữ cảnh phủ định kéo biểu diễn về hai phía khác nhau). Vector `[CLS]` ở lớp cuối là một biểu diễn 768 chiều có sẵn, và một siêu phẳng tuyến tính (head) đã tách được hai lớp ở mức 79%, cao hơn rõ rệt mức đoán 50% và cao hơn cả small-scratch được train toàn bộ (74.8%). Nghĩa là thông tin sentiment đã *tách tuyến tính tương đối* trong biểu diễn pretrained, dù encoder chưa từng thấy nhãn.
- **Vì sao fine-tuning cải thiện?** Frozen bị giới hạn bởi cái encoder đã học cho mục tiêu khác: `[CLS]` của DistilBERT không được huấn luyện trực tiếp cho classification (DistilBERT còn bỏ NSP), nên nó chưa gom thông tin sentiment tốt. Fine-tuning cập nhật toàn bộ tham số để (i) làm cho `[CLS]` tập hợp thông tin cảm xúc từ các token, (ii) điều chỉnh attention để chú ý đến từ/ngữ cảnh đảo nghĩa (phủ định, "but"), (iii) thích nghi từ vựng với phong cách review phim. Bằng chứng: val loss/acc và test acc đều cao hơn; chưa có thí nghiệm tách riêng từng cơ chế nên (i)-(iii) vẫn là giả thuyết.
- *Cost:* frozen có bộ nhớ đỉnh 0.44 GB so với 1.35 GB (không lưu activation/gradient/optimizer state cho encoder). Thời gian: 378 s vs 1,118 s nhưng bị nhiễu tranh chấp GPU như đã nói. Về lý thuyết backward qua encoder chiếm ~2/3 chi phí mỗi bước nên frozen rẻ hơn khoảng 3 lần mỗi bước.
- *Khớp dự đoán:* chọn **C (kém hơn)** là đúng; kỳ vọng frozen ≈ 80–87% thì thực tế 79.2% — nằm sát đáy khoảng dự đoán.

## 4. Experiment 3 — Amount of training data (data-efficiency curve)

In [6]:
fracs = [10, 25, 50, 100]
def get(kind, f):
    nm = {"ft": f"ft_len128_{f}", "frozen": f"frozen_len128_{f}", "scratch": f"scratch_small_len128_{f}"}[kind]
    return res[nm]["metrics"]["imdb_test"]
rows = []
for f in fracs:
    d = {"Training data": f"{f}% ({int(10000*f/100):,})"}
    for kind, lab in [("ft", "DistilBERT fine-tune"), ("frozen", "DistilBERT frozen"), ("scratch", "Small scratch")]:
        m = get(kind, f); d[f"{lab} Acc"] = m["accuracy"]; d[f"{lab} F1"] = m["f1"]
    rows.append(d)
exp3 = pd.DataFrame(rows); display(exp3.round(4))
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
styles = {"ft": (BLUE, "o", "DistilBERT pretrained + fine-tune"), "frozen": (GREEN, "s", "DistilBERT pretrained, frozen"), "scratch": (ORANGE, "^", "Transformer nhỏ from scratch")}
for ax, key, ttl in zip(axes, ["accuracy", "f1"], ["Accuracy", "F1"]):
    for kind, (c, mk, lab) in styles.items():
        y = [get(kind, f)[key] for f in fracs]
        ax.plot(fracs, y, marker=mk, color=c, label=lab, lw=2)
        for x_, y_ in zip(fracs, y): ax.annotate(f"{y_:.3f}", (x_, y_), textcoords="offset points", xytext=(0, 6), ha="center", fontsize=7, color=c)
    ax.set_xscale("log"); ax.set_xticks(fracs); ax.set_xticklabels([f"{f}%" for f in fracs])
    ax.set_xlabel("% training data (log scale; 100% = 10,000 review)"); ax.set_ylabel(f"Test {ttl} (IMDb subset 5k)"); ax.set_title(f"Data-efficiency curve — {ttl}"); ax.grid(alpha=.3)
axes[0].legend(loc="lower right", fontsize=8)
fig.tight_layout(); fig.savefig("data_efficiency.png", dpi=150); plt.close(fig)
ctrl = res["ft_len128_10_ep10"]["metrics"]["imdb_test"]; base10 = get("ft", 10)
print(f"Control: 10% data, 2 epoch (64 bước): acc={base10['accuracy']:.4f} F1={base10['f1']:.4f}  |  10% data, 10 epoch (320 bước): acc={ctrl['accuracy']:.4f} F1={ctrl['f1']:.4f}  | 100% data, 2 epoch (626 bước): acc={get('ft',100)['accuracy']:.4f}")
drop = pd.DataFrame({k: [get(k, 100)["accuracy"] - get(k, 10)["accuracy"], get(k, 100)["accuracy"] - get(k, 25)["accuracy"]] for k in ["ft", "frozen", "scratch"]}, index=["Acc(100%) - Acc(10%)", "Acc(100%) - Acc(25%)"])
display(drop.round(4))

,Training data,DistilBERT fine-tune Acc,DistilBERT fine-tune F1,DistilBERT frozen Acc,DistilBERT frozen F1,Small scratch Acc,Small scratch F1
0,"10% (1,000)",0.7968,0.7979,0.7332,0.7385,0.5948,0.5728
1,"25% (2,500)",0.8448,0.8416,0.7658,0.7571,0.6742,0.7086
2,"50% (5,000)",0.8584,0.8602,0.7778,0.7674,0.7186,0.7315
3,"100% (10,000)",0.8704,0.8701,0.7918,0.7889,0.7484,0.7328


Control: 10% data, 2 epoch (64 bước): acc=0.7968 F1=0.7979  |  10% data, 10 epoch (320 bước): acc=0.8180 F1=0.8277  | 100% data, 2 epoch (626 bước): acc=0.8704


,ft,frozen,scratch
Acc(100%) - Acc(10%),0.0736,0.0586,0.1536
Acc(100%) - Acc(25%),0.0256,0.0260,0.0742


![data efficiency](data_efficiency.png)

✍️ **Observation → Evidence → Interpretation (Exp 3).**
- *Observation:* khi giảm từ 100% (10,000 review) xuống 10% (1,000 review), accuracy của DistilBERT fine-tune giảm 87.0 → 79.7 (**−7.4 điểm**), frozen 79.2 → 73.3 (−5.9), small scratch 74.8 → 59.5 (**−15.4**; F1 0.733 → 0.573). Từ 25% lên 100%, pretrained chỉ tăng 2.6 điểm (0.845 → 0.870) còn scratch tăng 7.4 điểm (0.674 → 0.748): đường scratch dốc hơn, đường pretrained thoải hơn và luôn nằm trên (0.797 / 0.845 / 0.858 / 0.870 so với 0.595 / 0.674 / 0.719 / 0.748).
- *Evidence cần thận trọng:* (a) ở mức 10%, model fine-tune chỉ được cập nhật 64 bước (2 epoch × 32 batch); khi cho thêm số bước (10 epoch = 320 bước, run control) accuracy tăng 79.7 → **81.8** (F1 0.798 → 0.828), nên một phần "tụt" ở 10% là *train chưa đủ bước* chứ không chỉ *thiếu dữ liệu*; tuy vậy vẫn kém 5.2 điểm so với 100%. (b) Scratch dùng 6 epoch cho mọi mức; nó có thể cũng chưa tối ưu. (c) Mỗi điểm chỉ 1 seed, test n=5,000 (sai số ≈ ±1 điểm), các chênh lệch nhỏ (ví dụ 50% vs 100% của scratch) không nên đọc quá mức.
- *Interpretation:* kết luận dựa trên toàn đường cong chứ không một điểm: pretrained **luôn** cao hơn scratch ở mọi mức dữ liệu và khoảng cách lớn nhất ở mức ít dữ liệu nhất (+20.2 điểm ở 10%, +12.2 ở 100%). Điều này nhất quán với giả thuyết rằng pretraining "trả trước" việc học biểu diễn nên model cần ít nhãn hơn. Frozen cho thấy một điều khác: mất ít nhất (−5.9) nhưng cũng thấp nhất trong hai model pretrained ở mọi mức, vì capacity task-specific của nó chỉ là 1,538 tham số — ít dữ liệu không làm nó overfit, nhưng nhiều dữ liệu cũng không giúp nó vượt trần của biểu diễn cố định.
- **Trả lời:** có, pretraining giúp dùng dữ liệu downstream hiệu quả hơn (đường cong cao hơn và thoải hơn so với scratch), với lưu ý về 1 seed và về số bước train ở mức 10%.

## 5. Experiment 4 — Maximum sequence length (50% train = 5,000 review, 2 epoch, cùng split/test; xem README về lý do dùng 50%)

In [7]:
rows = []
for T in (64, 128, 256):
    r = res[f"ft_len{T}_50"]; m = r["metrics"]["imdb_test"]
    rows.append({"Max length": T, "Accuracy": m["accuracy"], "F1": m["f1"], "Train time (s)": r["train_time_s"], "Time/epoch (s)": r["time_per_epoch_s"], "Peak GPU mem in run (MB)": r["peak_gpu_mem_mb"], "micro-batch": r["batch_size"] // r["accum_steps"]})
exp4 = pd.DataFrame(rows); display(exp4.round(4))

,Max length,Accuracy,F1,Train time (s),Time/epoch (s),Peak GPU mem in run (MB),micro-batch
0,64,0.7932,0.7951,102.5529,47.9365,1324.1162,32
1,128,0.8584,0.8602,172.6191,79.8921,1572.2529,16
2,256,0.8970,0.8982,212.6540,100.3816,1474.7954,4


In [8]:
# Benchmark có kiểm soát (cùng micro-batch = 4, dữ liệu giả toàn token thật, forward+backward, fp16): time/step và peak memory theo T
bpath = OUT / "seqlen_benchmark.csv"
if bpath.exists():
    bench = pd.read_csv(bpath)
else:
    bench = pc.benchmark_seq_len(); bench.to_csv(bpath, index=False)
display(bench.round(4))
# Ước lượng FLOPs lý thuyết / layer / sequence (d=768, FFN=4d): projections 8·T·d² + FFN 16·T·d² + attention (QK^T và AV) 4·T²·d
d = 768
fl = pd.DataFrame({"max_len": bench["max_len"]})
fl["linear part (8Td²+16Td²)"] = 24 * fl["max_len"] * d**2
fl["attention T² part (4T²d)"] = 4 * fl["max_len"]**2 * d
fl["% attention T² part"] = 100 * fl["attention T² part (4T²d)"] / (fl["linear part (8Td²+16Td²)"] + fl["attention T² part (4T²d)"])
fl["total FLOPs rel. to T=64"] = (fl["linear part (8Td²+16Td²)"] + fl["attention T² part (4T²d)"]) / (fl["linear part (8Td²+16Td²)"] + fl["attention T² part (4T²d)"]).iloc[0]
fl["total FLOPs rel. to T=64"] = (fl["linear part (8Td²+16Td²)"] + fl["attention T² part (4T²d)"]) / ((fl["linear part (8Td²+16Td²)"] + fl["attention T² part (4T²d)"]).iloc[0])
display(fl.round(2))
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.plot(bench["max_len"], fl["total FLOPs rel. to T=64"], "s-", color=GREEN, label="FLOPs lý thuyết")
ax.plot(bench["max_len"], bench["time_rel_to_64"], "o-", color=BLUE, label="thời gian / step (đo thực tế)")
ax.plot(bench["max_len"], bench["max_len"] / 64, "--", color=GREY, label="tuyến tính  O(T)")
ax.plot(bench["max_len"], bench["T2_rel_to_64"], ":", color=RED, label="bậc hai  O(T²)")
ax.set_xscale("log", base=2); ax.set_yscale("log", base=2); ax.set_xticks(bench["max_len"]); ax.set_xticklabels(bench["max_len"])
ax.set_xlabel("max sequence length T"); ax.set_ylabel("chi phí tương đối so với T=64"); ax.set_title("Chi phí theo T (DistilBERT, micro-batch 4)"); ax.legend(fontsize=8); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig("sequence_length_cost.png", dpi=150); plt.close(fig)

,max_len,batch_size,sec_per_step,peak_gpu_mem_mb,time_rel_to_64,T2_rel_to_64
0,64,4,0.1519,1291.0889,1.0000,1.0
1,128,4,0.1650,1293.4678,1.0866,4.0
2,256,4,0.2123,1349.4463,1.3978,16.0
3,512,4,0.3063,1560.3447,2.0163,64.0


,max_len,linear part (8Td²+16Td²),attention T² part (4T²d),% attention T² part,total FLOPs rel. to T=64
0,64,905969664,12582912,1.37,1.00
1,128,1811939328,50331648,2.70,2.03
2,256,3623878656,201326592,5.26,4.16
3,512,7247757312,805306368,10.00,8.77


![seq length cost](sequence_length_cost.png)

✍️ **Observation → Evidence → Interpretation (Exp 4).**
- *Setup:* để tiết kiệm GPU, Exp 4 dùng 50% training pool (5,000 review, 2 epoch, giống hệt run `ft_len128_50` của Exp 3 cho T=128), chỉ đổi `max_len`.
- *Observation:* accuracy **0.793 / 0.858 / 0.897** và F1 0.795 / 0.860 / 0.898 cho T = 64 / 128 / 256. Mỗi lần gấp đôi T tăng accuracy +6.5 rồi +3.9 điểm: có giảm dần nhưng chưa bão hòa.
- *Evidence:* IMDb dài (median 222 token, mean 297; chỉ 12% review ≤128 token; xem `tokenization.ipynb` mục 5). Ở T=64 chỉ giữ ~21% tổng token, T=128 ~41%, T=256 ~69%. Accuracy theo độ dài review (model T=128): ≤128 token: 0.912; 128–256: 0.894; 256–512: 0.847; >512: 0.795 — review càng dài, càng nhiều nội dung bị cắt và accuracy càng thấp.
- *Cost:* benchmark có kiểm soát (cùng micro-batch 4, dữ liệu giả toàn token thật): thời gian/step 0.152 → 0.165 → 0.212 → 0.306 s cho T = 64 → 512, tức 1.00 / 1.09 / 1.40 / 2.02 lần; bộ nhớ đỉnh 1.29 → 1.56 GB. Chi phí đo thực tế tăng **chậm hơn nhiều** so với `O(T²)` (×4, ×16, ×64) và chậm hơn cả tuyến tính (×2, ×4, ×8). Lý do (tính từ FLOPs ở bảng trên): với d=768 phần attention `4T²d` chỉ chiếm 1.4% (T=64), 5.3% (T=256), 10% (T=512) tổng FLOPs/layer; phần còn lại (projection + FFN) tuyến tính theo T. Ngoài ra batch 4 × T=64 là quá nhỏ để GPU bão hòa nên các số đo ở T nhỏ bị chi phối bởi overhead/launch và bị nhiễu do GPU dùng chung. **Thời gian trong bảng "Train time" của từng run không dùng để kết luận** (micro-batch khác nhau, GPU bị tranh chấp).
- **Tăng sequence length có luôn làm model tốt hơn không?** Trong phạm vi này thì có (T ≤ 256 vẫn tăng, vì IMDb dài và review hay chốt đánh giá ở cuối), nhưng *không phải luôn luôn*: lợi ích phụ thuộc vào việc phần bị cắt còn chứa thông tin có ích. Với review ngắn (Amazon: median 94 token, Rotten Tomatoes: 27 token) tăng T không thêm gì mà chỉ tốn chi phí (và padding). Chưa thử T=512 trong phần train (chỉ benchmark), nên chưa biết điểm bão hòa.
- *Khớp dự đoán:* đúng chiều (acc tăng theo T, chi phí tăng ít hơn bậc hai), nhưng mình dự đoán "gain giảm dần rõ rệt" còn kết quả cho thấy gain vẫn lớn ở 128 → 256.

## 6. Experiment 5 — Domain shift (train trên IMDb, test trên domain khác)

In [9]:
names = [("ft_len128_100", "DistilBERT fine-tuned"), ("frozen_len128_100", "DistilBERT frozen"), ("scratch_small_len128_100", "Small scratch"), ("bert_random_len128_100", "DistilBERT random init")]
sets = [("imdb_test", "IMDb test (in-domain)"), ("rotten_tomatoes", "Rotten Tomatoes (phim, câu ngắn)"), ("amazon", "Amazon (product reviews)")]
rows = []
for nm, lab in names:
    d = {"Model": lab}
    for k, sl in sets:
        d[f"Acc {k}"] = res[nm]["metrics"][k]["accuracy"]
    for k, sl in sets[1:]:
        d[f"Δ {k}"] = res[nm]["metrics"][k]["accuracy"] - res[nm]["metrics"]["imdb_test"]["accuracy"]
    rows.append(d)
exp5 = pd.DataFrame(rows); display(exp5.round(4))
# F1 / recall / precision của model fine-tuned
display(pd.DataFrame({k: {m: res["ft_len128_100"]["metrics"][k][m] for m in ["accuracy", "precision", "recall", "f1", "fp", "fn", "n"]} for k, _ in sets}).T.round(4))
fig, ax = plt.subplots(figsize=(8, 3.8)); w = 0.2
for i, (nm, lab) in enumerate(names):
    ys = [res[nm]["metrics"][k]["accuracy"] for k, _ in sets]
    ax.bar(np.arange(len(sets)) + (i - 1.5) * w, ys, w, label=lab, color=[BLUE, GREEN, ORANGE, GREY][i])
    for x_, y_ in zip(np.arange(len(sets)) + (i - 1.5) * w, ys): ax.text(x_, y_ + .005, f"{y_:.2f}", ha="center", fontsize=7)
ax.set_xticks(range(len(sets))); ax.set_xticklabels([s for _, s in sets], fontsize=8); ax.set_ylim(0.4, 1.0); ax.set_ylabel("Accuracy"); ax.set_title("Domain shift: train IMDb -> test trên 3 tập"); ax.legend(fontsize=7, loc="upper center", ncol=4, bbox_to_anchor=(0.5, -0.12)); ax.grid(axis="y", alpha=.3)
fig.tight_layout(); fig.savefig("domain_shift.png", dpi=150, bbox_inches="tight"); plt.close(fig)

,Model,Acc imdb_test,Acc rotten_tomatoes,Acc amazon,Δ rotten_tomatoes,Δ amazon
0,DistilBERT fine-tuned,0.8704,0.7908,0.9135,-0.0796,0.0431
1,DistilBERT frozen,0.7918,0.7411,0.8240,-0.0507,0.0322
2,Small scratch,0.7484,0.6098,0.7520,-0.1386,0.0036
3,DistilBERT random init,0.8000,0.7008,0.7730,-0.0992,-0.0270


,accuracy,precision,recall,f1,fp,fn,n
imdb_test,0.8704,0.8719,0.8684,0.8701,319.0,329.0,5000.0
rotten_tomatoes,0.7908,0.7476,0.8780,0.8076,158.0,65.0,1066.0
amazon,0.9135,0.9082,0.9200,0.9141,93.0,80.0,2000.0


![domain shift](domain_shift.png)

In [10]:
# Diễn giải domain shift cần kiểm soát độ dài/độ khó: thống kê độ dài token và calibration trên từng tập (model fine-tuned)
_, _, _te = pc.load_imdb_splits(); dsets = pc.load_domain_sets(); dsets["imdb_test"] = _te.iloc[pc.test_subset_index(_te)]
rows = []
for k in ["imdb_test", "rotten_tomatoes", "amazon"]:
    L = np.array([len(x) for x in tok(dsets[k]["text"].tolist(), truncation=False)["input_ids"]])
    Pk = probs("ft_len128_100")[k]; ck = Pk.max(1); yk = dsets[k]["label"].to_numpy(); pk = Pk.argmax(1)
    ece_k = 0; ed = np.linspace(0.5, 1, 11)
    for lo, hi in zip(ed[:-1], ed[1:]):
        sel = (ck >= lo) & (ck < hi + (1e-9 if hi == 1 else 0))
        if sel.any(): ece_k += sel.mean() * abs((pk[sel] == yk[sel]).mean() - ck[sel].mean())
    rows.append({"set": k, "n": len(L), "median tokens": np.median(L), "mean tokens": L.mean().round(1), "% > 128 tokens (bị cắt)": 100 * (L > 128).mean(), "accuracy": (pk == yk).mean(), "mean confidence": ck.mean(), "ECE": ece_k, "predicted positive rate": pk.mean()})
display(pd.DataFrame(rows).round(3))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (725 > 512). Running this sequence through the model will result in indexing errors


,set,n,median tokens,mean tokens,% > 128 tokens (bị cắt),accuracy,mean confidence,ECE,predicted positive rate
0,imdb_test,5000,220.0,293.0,87.72,0.870,0.863,0.019,0.498
1,rotten_tomatoes,1066,27.0,27.7,0.00,0.791,0.812,0.042,0.587
2,amazon,2000,94.0,102.5,30.55,0.914,0.861,0.053,0.506


✍️ **Observation → Evidence → Interpretation (Exp 5).**
- *Observation:* model fine-tune trên IMDb: in-domain **0.870**; Rotten Tomatoes (phim, câu rất ngắn) **0.791** (−8.0 điểm); Amazon (product reviews) **0.914** (+4.3 điểm — **cao hơn** in-domain!). Small-scratch: 0.748 → 0.610 (RT, −13.9) và 0.752 (Amazon, +0.4). Frozen: −5.1 (RT), +3.2 (Amazon).
- *Evidence & cảnh báo:* con số "Amazon cao hơn IMDb" **không** có nghĩa là không có domain shift. IMDb test bị cắt ở 128 token (88% review > 128 token, mean 293) còn Amazon ngắn (median 94, 31% > 128) và hay có từ cảm xúc rõ ràng ("love it", "waste of money"), nên độ khó của hai tập khác nhau; so sánh accuracy tuyệt đối giữa tập khác nhau bị lẫn giữa *độ khó/độ dài* và *shift*. Để đo shift đúng cần baseline in-domain train trên chính Amazon/RT — lab này chưa làm. Phần có thể kết luận mạnh hơn: (i) trên RT có suy giảm rõ −8.0 điểm cho fine-tuned và −13.9 cho scratch; (ii) model fine-tuned bị **lệch dự đoán positive** trên RT: precision 0.748, recall 0.878, tỉ lệ dự đoán positive 58.7% trong khi nhãn cân bằng 50% (FP=158, FN=65) — đây là bằng chứng của shift về phân phối (câu ngắn, giọng điệu kiểu nhà phê bình), không phải do độ khó đơn thuần; (iii) calibration xấu đi khi shift: ECE 0.019 (in-domain) → 0.042 (RT) → 0.053 (Amazon), và trên RT confidence trung bình 0.81 vs accuracy 0.79.
- **Pretrained model có nghĩa là đã "hiểu mọi domain" không?** Không. Pretrained fine-tuned vẫn mất 8 điểm và bị lệch nhãn trên RT; nó chỉ mất *ít hơn* scratch (−8.0 vs −13.9) và random-init (−9.9) — pretraining cho biểu diễn tổng quát hơn (vừa giúp trên Amazon, mà model chưa từng thấy một review sản phẩm nào lúc fine-tune: 91.4%), nhưng không đảm bảo bền vững khi style thay đổi. Nguyên nhân khả dĩ của suy giảm trên RT (giả thuyết, chưa kiểm chứng riêng từng yếu tố): câu rất ngắn khác phân phối độ dài lúc train (median 27 vs 220 token), từ vựng/phong cách phê bình ("a fascinating but flawed..."), nhãn được gán theo kiểu khác (snippet cắt từ bài phê bình), và head đã học tương quan đặc thù IMDb.
- *Khớp dự đoán:* dự đoán "Amazon giảm nhiều hơn RT" **sai** (Amazon tăng, RT giảm); dự đoán "scratch giảm nhiều hơn pretrained" đúng ở RT, còn ở Amazon cả hai đều không giảm.

## 7. Application — Sentiment classification (DistilBERT fine-tuned, IMDb test subset 5,000)

In [11]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix, classification_report
train_df, val_df, test_all = pc.load_imdb_splits()
test_df = test_all.iloc[pc.test_subset_index(test_all)]          # giữ nguyên index gốc của IMDb test; đúng thứ tự với xác suất đã lưu
y = test_df["label"].to_numpy(); P = probs("ft_len128_100")["imdb_test"]; pred = P.argmax(1); conf = P.max(1)
m = res["ft_len128_100"]["metrics"]["imdb_test"]
print(pd.Series({k: round(m[k], 4) for k in ["accuracy", "precision", "recall", "f1", "loss"]}).to_string())
print(classification_report(y, pred, target_names=["negative", "positive"], digits=4))
cm = confusion_matrix(y, pred)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
ConfusionMatrixDisplay(cm, display_labels=["negative", "positive"]).plot(ax=axes[0], cmap="Blues", colorbar=False, values_format="d"); axes[0].set_title("Confusion matrix (đếm)")
ConfusionMatrixDisplay(confusion_matrix(y, pred, normalize="true"), display_labels=["negative", "positive"]).plot(ax=axes[1], cmap="Blues", colorbar=False, values_format=".3f"); axes[1].set_title("Chuẩn hóa theo hàng (recall)")
fig.suptitle("DistilBERT fine-tuned — IMDb test subset (n=5,000, max_len=128)"); fig.tight_layout(); fig.savefig("confusion_matrix.png", dpi=150); plt.close(fig)
print("TN, FP, FN, TP =", cm.ravel().tolist())
demo = pd.DataFrame(res["ft_len128_100"]["demo_preds"]); demo["pred"] = np.where(demo["p_pos"] > 0.5, "positive", "negative")
display(demo.round(4))

accuracy     0.8704
precision    0.8719
recall       0.8684
f1           0.8701
loss         0.3063
              precision    recall  f1-score   support

    negative     0.8689    0.8724    0.8707      2500
    positive     0.8719    0.8684    0.8701      2500

    accuracy                         0.8704      5000
   macro avg     0.8704    0.8704    0.8704      5000
weighted avg     0.8704    0.8704    0.8704      5000

TN, FP, FN, TP = [2181, 319, 329, 2171]


,text,p_pos,pred
0,This movie is great.,0.9532,positive
1,I do not like this movie.,0.0547,negative
2,The film is not bad at all.,0.1267,negative
3,"The plot was predictable, but the acting was a...",0.9041,positive
4,"Oh great, another two hours of my life I will ...",0.4035,negative
5,"I expected to hate it, yet I ended up loving e...",0.8424,positive
6,The product broke after two days. Terrible qua...,0.0379,negative
7,"The movie was not good, not even close to bein...",0.0266,negative


![confusion matrix](confusion_matrix.png)

✍️ **Nhận xét (Application).** DistilBERT fine-tuned: accuracy 0.870, precision 0.872, recall 0.868, F1 0.870 trên 5,000 review (cân bằng 2,500/2,500). Confusion matrix: TN=2181, FP=319, FN=329, TP=2171 → lỗi gần đối xứng (FP 319 vs FN 329), không có thiên lệch lớp rõ ràng trên in-domain (khác với RT ở Exp 5, nơi FP ≫ FN). Chỉ báo accuracy sẽ che mất sự mất cân bằng đó, nên cần precision/recall/confusion matrix.
Bảng 8 câu thủ công (chỉ là minh họa, không phải đánh giá thống kê): câu rõ ràng đúng (*great* → 0.95, *do not like* → 0.05, *not good, not even close* → 0.03); **"The film is not bad at all."** bị dự đoán negative (P(pos)=0.13) dù ý là khen — một dạng phủ định kép/litotes mà model chưa xử lý; câu mỉa mai *"Oh great, another two hours..."* P(pos)=0.40, tức đúng hướng (negative) nhưng chỉ vừa phải tự tin; câu "yet I ended up loving every minute" (đảo chiều giữa câu) đúng (0.84).

## 8. Confidence analysis

In [12]:
bins = [("Low (0.5-0.7)", 0.5, 0.7), ("Medium (0.7-0.9)", 0.7, 0.9), ("High (0.9-1.0)", 0.9, 1.0001)]
rows = []
for lab, lo, hi in bins:
    sel = (conf >= lo) & (conf < hi)
    rows.append({"Bin": lab, "n": int(sel.sum()), "% of preds": 100 * sel.mean(), "Accuracy in bin": (pred[sel] == y[sel]).mean() if sel.any() else np.nan, "Mean confidence": conf[sel].mean() if sel.any() else np.nan, "# errors": int((pred[sel] != y[sel]).sum())})
cb = pd.DataFrame(rows); display(cb.round(4))
very = conf >= 0.99
print(f"conf >= 0.99: n={very.sum()} ({100*very.mean():.1f}%), accuracy={(pred[very]==y[very]).mean():.4f}, #errors={(pred[very]!=y[very]).sum()}")
print(f"Mean confidence correct={conf[pred==y].mean():.4f} | wrong={conf[pred!=y].mean():.4f}")
ece = 0; edges = np.linspace(0.5, 1, 11)
for lo, hi in zip(edges[:-1], edges[1:]):
    s = (conf >= lo) & (conf < hi + (1e-9 if hi == 1 else 0))
    if s.any(): ece += s.mean() * abs((pred[s] == y[s]).mean() - conf[s].mean())
print(f"ECE (10 bin) = {ece:.4f}")
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
axes[0].hist(conf[pred == y], bins=25, alpha=.75, color=BLUE, label="đúng"); axes[0].hist(conf[pred != y], bins=25, alpha=.85, color=RED, label="sai")
axes[0].set_yscale("log"); axes[0].set_xlabel("confidence = max softmax"); axes[0].set_ylabel("số mẫu (log)"); axes[0].set_title("Phân bố confidence"); axes[0].legend()
axes[1].bar(cb["Bin"], cb["Accuracy in bin"], color=[RED, ORANGE, BLUE]); axes[1].set_ylim(0, 1.15); axes[1].set_ylabel("accuracy trong bin"); axes[1].set_title("Accuracy theo mức confidence")
for i, (a_, n_) in enumerate(zip(cb["Accuracy in bin"], cb["n"])): axes[1].text(i, a_ + .01, f"{a_:.3f}\n(n={n_})", ha="center", fontsize=8)
axes[1].tick_params(axis="x", labelsize=8)
acc_b, conf_b, cnt_b = [], [], []
for lo, hi in zip(edges[:-1], edges[1:]):
    s = (conf >= lo) & (conf < hi + (1e-9 if hi == 1 else 0))
    if s.sum() >= 5: acc_b.append((pred[s] == y[s]).mean()); conf_b.append(conf[s].mean()); cnt_b.append(s.sum())
axes[2].plot([0.5, 1], [0.5, 1], "--", color=GREY, label="hiệu chỉnh hoàn hảo"); axes[2].plot(conf_b, acc_b, "o-", color=BLUE, label="DistilBERT fine-tuned")
axes[2].set_xlabel("mean confidence"); axes[2].set_ylabel("accuracy"); axes[2].set_title(f"Reliability diagram (ECE={ece:.3f})"); axes[2].legend(); axes[2].grid(alpha=.3)
fig.tight_layout(); fig.savefig("confidence_analysis.png", dpi=150); plt.close(fig)

,Bin,n,% of preds,Accuracy in bin,Mean confidence,# errors
0,Low (0.5-0.7),803,16.06,0.6202,0.6031,305
1,Medium (0.7-0.9),1366,27.32,0.8089,0.8150,261
2,High (0.9-1.0),2831,56.62,0.9710,0.9594,82


conf >= 0.99: n=64 (1.3%), accuracy=1.0000, #errors=0
Mean confidence correct=0.8836 | wrong=0.7224
ECE (10 bin) = 0.0191


![confidence](confidence_analysis.png)

✍️ **Observation → Evidence → Interpretation (Confidence).**
- *Observation:* 3 mức confidence (max softmax): Low (0.5–0.7) có accuracy **0.620** (n=803, 16%); Medium (0.7–0.9) **0.809** (n=1,366, 27%); High (0.9–1.0) **0.971** (n=2,831, 57%). Confidence trung bình của dự đoán đúng 0.884 so với 0.722 của dự đoán sai. ECE (10 bin) = 0.019 — in-domain model khá được hiệu chỉnh (reliability diagram bám sát đường chéo).
- *Evidence:* confidence **có** mang thông tin: accuracy tăng đơn điệu theo mức confidence, và 64 mẫu có confidence ≥ 0.99 đều đúng (nhưng 64 mẫu là ít, chỉ 1.3% tổng).
- **Confidence cao có đồng nghĩa với prediction đúng không?** Không. Trong bin High vẫn có **82 lỗi** (2.9% bin), và 23 lỗi có confidence ≥ 0.95 trên tổng 648 lỗi; hai lỗi tự tin nhất (0.986) mà mình xem lại không hẳn là lỗi model (xem `error_analysis.md`: một nhãn dường như nhiễu, một nhãn mơ hồ). Confidence còn xấu đi khi dữ liệu lệch domain (ECE 0.042–0.053 trên RT/Amazon, Exp 5). Tức `confidence ≠ correctness`: nó là ước lượng của model, đáng tin hơn trên cùng phân phối, và không thể thay thế kiểm tra đúng-sai bằng nhãn thật.

## 9. Dữ liệu cho Error Analysis (≥3 FP, ≥3 FN, ≥2 sai nhưng rất tự tin)

In [13]:
rng = np.random.RandomState(42)
test_df = test_df.copy(); test_df["pred"] = pred; test_df["p_pos"] = P[:, 1]; test_df["conf"] = conf
tok_len = np.array([len(x) for x in tok(test_df["text"].tolist(), truncation=False)["input_ids"]]); test_df["n_tokens"] = tok_len
fp_all = test_df[(test_df.label == 0) & (test_df.pred == 1)]; fn_all = test_df[(test_df.label == 1) & (test_df.pred == 0)]
hi_wrong = test_df[(test_df.label != test_df.pred)].sort_values("conf", ascending=False)
top_fp, top_fn = hi_wrong[hi_wrong.pred == 1].head(1), hi_wrong[hi_wrong.pred == 0].head(1)     # 2 mẫu sai tự tin nhất (1 FP, 1 FN)
rest_fp, rest_fn = fp_all.drop(top_fp.index), fn_all.drop(top_fn.index)
pick_fp, pick_fn = rest_fp.sample(3, random_state=42), rest_fn.sample(3, random_state=42)           # 3 FP + 3 FN ngẫu nhiên (seed 42)
samples = pd.concat([pick_fp.assign(kind="FP"), pick_fn.assign(kind="FN"), top_fp.assign(kind="FP (very confident)"), top_fn.assign(kind="FN (very confident)")])
samples.index.name = "test_idx"
samples.reset_index()[["test_idx", "kind", "label", "pred", "p_pos", "conf", "n_tokens", "text"]].to_csv("error_samples.csv", index=False, encoding="utf-8")
print(f"Tổng lỗi = {(pred != y).sum()} (FP={len(fp_all)}, FN={len(fn_all)}); lỗi có conf >= 0.95: {(hi_wrong.conf >= 0.95).sum()}")
for idx, r in samples.iterrows():
    print(f"\n#{idx} [{r.kind}] true={'pos' if r.label else 'neg'} pred={'pos' if r.pred else 'neg'} P(pos)={r.p_pos:.3f} conf={r.conf:.3f} tokens={r.n_tokens}")
    print("   ", r.text[:600].replace("\n", " "))

Tổng lỗi = 648 (FP=319, FN=329); lỗi có conf >= 0.95: 23

#2766 [FP] true=neg pred=pos P(pos)=0.750 conf=0.750 tokens=189
    This movie is a great movie ONLY if you need something to sit and laugh at the stupidity of it. As a geologist this movie gets most of the important facts wrong and uses actors that are too young to even be considered in the top of their fields. It is interesting how it shows spurting lava in massive caverns below the Earth's surface. It also is funny how seismically active areas are shown to have massive destruction from a 6.5 magnitude earthquake. They seem to forget the building standards in these areas would be higher needing a bigger quake to do this much damage. Also it is funny how much

#10459 [FP] true=neg pred=pos P(pos)=0.711 conf=0.711 tokens=102
    This sickly sweet and laboriously paced 5-reeler is definitely not among Harold Lloyd's better films. Gags are sparse and mostly uninspired. Saccharine melodrama is abundant. The setup takes forever, as 

In [14]:
# Thống kê lỗi theo độ dài: có phải lỗi tập trung ở review bị truncation?
test_df["truncated"] = test_df["n_tokens"] > 128
print(test_df.groupby("truncated").apply(lambda g: pd.Series({"n": len(g), "accuracy": (g.label == g.pred).mean()})).round(4))
test_df["len_bin"] = pd.cut(test_df["n_tokens"], [0, 128, 256, 512, 10000])
print(test_df.groupby("len_bin").apply(lambda g: pd.Series({"n": len(g), "accuracy": (g.label == g.pred).mean()})).round(4))

                n  accuracy
truncated                  
False       614.0    0.9121
True       4386.0    0.8646
                   n  accuracy
len_bin                       
(0, 128]       614.0    0.9121
(128, 256]    2354.0    0.8942
(256, 512]    1383.0    0.8467
(512, 10000]   649.0    0.7951


✍️ **Nhận xét.** Tổng 648 lỗi trên 5,000 (FP=319, FN=329). Accuracy giảm theo độ dài review: 0.912 (≤128 token, n=614) → 0.894 → 0.847 → 0.795 (>512 token, n=649). Đây là bằng chứng nhất quán với việc cắt ở 128 token làm mất phần kết luận của review dài (và Exp 4: T=256 tốt hơn T=128 +3.9 điểm), nhưng độ dài cũng tương quan với độ khó/mơ hồ của review nên đây chưa phải quan hệ nhân quả. Phân tích từng mẫu (loại lỗi, giả thuyết) nằm trong `error_analysis.md`.